# Hybrid RAG

This notebook continues from `02_7_vectordb_docker.ipynb` and teaches **hybrid retrieval** by progressively improving the RAG pipeline across three techniques:

1. **Keyword filter** — restrict the candidate pool to documents that contain a specific term
2. **Metadata filter** — use structured fields (score, year, artist) stored natively in ChromaDB
3. **LLM re-ranking** — retrieve a larger set, then let the model promote the most relevant

We run the **same sample query** at each stage so you can compare results directly.

**Prerequisites:** Complete `02_7` first — ChromaDB must be running with the `pitchfork_reviews` collection loaded.

## Setup

In [ ]:
%load_ext dotenv

import sys
sys.path.append('../../05_src/')
%dotenv ../../05_src/.secrets
%dotenv ../../05_src/.env

In [ ]:
from utils.clients import get_client
from IPython.display import display, Markdown
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import json
import os

client = get_client(use_gateway=False)
MODEL = os.getenv('MODEL')
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL')
COLLECTION_NAME = 'pitchfork_reviews'
CHROMA_URL = os.getenv('CHROMA_URL')

### Helper Functions

These functions are brought forward from `02_7`. `get_context_data()` queries ChromaDB and returns enriched result dicts (text + metadata). `generate_prompt()` builds the prompt with album context. `generate_response()` calls the model.

In [ ]:
def get_context_data(query: str, collection: chromadb.api.models.Collection, top_n: int,
                     where_document: dict = None, where: dict = None):
    kwargs = dict(query_texts=[query], n_results=top_n)
    if where_document:
        kwargs['where_document'] = where_document
    if where:
        kwargs['where'] = where
    results = collection.query(**kwargs)
    context_data = []
    for idx in range(len(results['ids'][0])):
        details = dict(results['metadatas'][0][idx])
        details['text'] = results['documents'][0][idx]
        context_data.append(details)
    return context_data

def generate_prompt(query: str, context_data: list):
    top_n = len(context_data)
    prompt = f'Given a query, provide a detailed response using the context from relevant Pitchfork reviews. The context will contain references to {top_n} album reviews.\n\n'
    prompt += 'The score is numeric and its scale is from 0 to 10, with 10 being the highest rating. Any album with a score greater than 8.0 is considered a must-listen; album with a score greater than 6.5 is good.\n\n'
    prompt += f'<query>{query}</query>\n\n'
    prompt += '<context>\n'
    for k, context in enumerate(context_data):
        prompt += f'<album {k}>\n'
        prompt += f"- Album Title: {context.get('album', 'N/A')}\n"
        prompt += f"- Album Artist: {context.get('artist', 'N/A')}\n"
        prompt += f"- Album Score: {context.get('score', 'N/A')}\n"
        prompt += f"- Genre: {context.get('genre', 'N/A')}\n"
        prompt += f"- Label: {context.get('label', 'N/A')}\n"
        prompt += f"- Release Year: {context.get('year', 'N/A')}\n"
        prompt += f"- Review Quote: {context.get('text', 'N/A')}\n"
        prompt += f'</album {k}>\n\n'
    prompt += '</context>\n\n'
    prompt += '\nBased on the context and nothing else, provide a detailed response to the query.'
    return prompt

def generate_response(query: str, collection: chromadb.api.models.Collection, top_n: int = 3,
                      where_document: dict = None, where: dict = None):
    context_data = get_context_data(query, collection, top_n,
                                    where_document=where_document, where=where)
    prompt = generate_prompt(query, context_data)
    response = client.responses.create(
        model=MODEL,
        instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
        input=[{'role': 'user', 'content': prompt}],
        max_output_tokens=500,
        temperature=0.7
    )
    return response.output_text

### Connect to ChromaDB

Set `USE_GATEWAY = True` if you are using the API gateway instead of a direct OpenAI key.

In [ ]:
USE_GATEWAY = os.getenv('USE_GATEWAY', 'False').lower() == 'true'

chroma = chromadb.HttpClient(host=CHROMA_URL)
if USE_GATEWAY:
    embedding_function = OpenAIEmbeddingFunction(
        api_key='any value',
        api_base='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
        api_type='openai',
        model_name=EMBEDDING_MODEL,
        default_headers={'x-api-key': os.getenv('API_GATEWAY_KEY')}
    )
else:
    embedding_function = OpenAIEmbeddingFunction(
        api_key=os.getenv('OPENAI_API_KEY'),
        model_name=EMBEDDING_MODEL
    )

collection = chroma.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)
print(f'Collection loaded: {collection.count():,} documents')

---

## Section 1: Baseline — Pure Vector Search

Before we improve retrieval, let's establish a baseline using the pure vector pipeline from `02_7`. We embed the query, find the 3 most similar review chunks by cosine similarity, and generate a response.

We will reuse the **same query** throughout every section so you can directly compare results.

In [ ]:
SAMPLE_QUERY = 'What are some highly rated albums by emerging indie artists?'

In [ ]:
baseline_response = generate_response(SAMPLE_QUERY, collection, top_n=3)
display(Markdown(baseline_response))

**What the baseline does:**
- Converts the query to a vector using the same embedding model as the corpus
- Returns the 3 chunks most similar by cosine distance

**What it doesn't do:**
- It has no knowledge of exact words in the query — `indie` is treated semantically, not literally
- It ignores structured fields like critic score or release year
- It returns the closest chunks by embedding distance, which may not be the *most useful* for the query

The next three sections address each of these gaps.

---

## Section 2: Keyword Filter (`where_document`)

ChromaDB's `where_document={"$contains": keyword}` filter restricts the search to documents whose **text contains the keyword** — before similarity ranking is applied.

This is useful when:
- The query involves **exact terms**: artist names, album titles, genre labels
- You need **lexical precision** over semantic breadth

It can hurt when:
- The keyword is rare or absent in the corpus → very few candidates remain
- Semantic synonyms are excluded (e.g., `indie` might miss `independent`, `lo-fi`)

In [ ]:
def keyword_search(query: str, collection: chromadb.api.models.Collection,
                   keyword: str, top_n: int = 3):
    """Vector search restricted to documents containing `keyword`."""
    return get_context_data(
        query, collection, top_n,
        where_document={'$contains': keyword}
    )

In [ ]:
keyword_results = keyword_search(SAMPLE_QUERY, collection, keyword='indie', top_n=3)

for i, r in enumerate(keyword_results):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

In [ ]:
keyword_response = generate_response(SAMPLE_QUERY, collection, top_n=3,
                                     where_document={'$contains': 'indie'})
display(Markdown(keyword_response))

**Compare with baseline:** The keyword filter ensures every result chunk contains the word `indie` literally. Depending on the corpus, this may surface different albums than the pure vector baseline — or the same ones, since `indie` is semantically close to the query.

**Try it:** Replace `'indie'` with an artist name (e.g., `'Radiohead'`, `'Arcade Fire'`) to see how keyword filtering can narrow results to a specific artist's reviews.

---

## Section 3: Metadata Filtering (`where`)

When we loaded the collection in `02_7`, we stored structured fields alongside each chunk:

| Field | Type | Example |
|-------|------|---------|
| `score` | float | `8.7` |
| `year` | int | `2001` |
| `artist` | str | `'Radiohead'` |
| `album` | str | `'Kid A'` |
| `genre` | str | `'Electronic, Rock'` |
| `label` | str | `'Capitol'` |

ChromaDB's `where` parameter filters on these fields **before** ranking. Supported operators: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$and`, `$or`, `$in`.

In [ ]:
def metadata_search(query: str, collection: chromadb.api.models.Collection,
                    where_filter: dict, top_n: int = 3):
    """Vector search restricted to documents matching `where_filter` on metadata fields."""
    return get_context_data(query, collection, top_n, where=where_filter)

**Example 1: Score filter** — only reviews rated above 8.0

In [ ]:
score_results = metadata_search(
    SAMPLE_QUERY, collection,
    where_filter={'score': {'$gt': 8.0}},
    top_n=3
)

for i, r in enumerate(score_results):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

In [ ]:
score_response = generate_response(SAMPLE_QUERY, collection, top_n=3,
                                   where={'score': {'$gt': 8.0}})
display(Markdown(score_response))

**Example 2: Compound filter** — high-scoring albums released from 2000 onwards

In [ ]:
compound_results = metadata_search(
    SAMPLE_QUERY, collection,
    where_filter={'$and': [{'score': {'$gt': 8.0}}, {'year': {'$gte': 2000}}]},
    top_n=3
)

for i, r in enumerate(compound_results):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

In [ ]:
compound_response = generate_response(
    SAMPLE_QUERY, collection, top_n=3,
    where={'$and': [{'score': {'$gt': 8.0}}, {'year': {'$gte': 2000}}]}
)
display(Markdown(compound_response))

**A note on genre filtering:**

You might expect to filter by genre with `{'genre': {'$eq': 'Rock'}}`. This works, but only for reviews with *exactly* that genre string. In our dataset, genres are stored as comma-separated strings (e.g., `'Electronic, Rock'`), so `$eq 'Rock'` won't match multi-genre entries.

This is a **schema design trade-off**: storing the primary genre as a separate field (`primary_genre`) would enable clean equality filtering. As a workaround, you can use `artist` or `year` filters (which are single-valued) alongside the semantic query.

**Try it:** Change the `score` threshold or `year` cutoff and observe how the candidate set changes. A very strict filter (e.g., `score > 9.5`) may return no results — the function returns an empty list gracefully.

---

## Section 4: LLM Re-ranking

Vector similarity finds documents that are *semantically close* to the query, but closeness in embedding space doesn't always mean *most useful for answering the question*.

**Re-ranking** separates retrieval from relevance scoring:
1. Retrieve a **larger candidate set** (e.g., top 10) with vector search
2. Ask the language model to **rank candidates by relevance** to the query
3. Return the top-k from the re-ranked list

This adds one API call but can meaningfully improve result quality — especially when the top-1 vector result is semantically close but not the most informative answer.

In [ ]:
def llm_rerank(context_data: list, query: str, top_k: int = 3) -> list:
    """Re-rank context_data by relevance to query using a single LLM call."""
    if not context_data:
        return context_data

    # Build a compact candidate list for the model
    candidates_text = ''
    for i, c in enumerate(context_data):
        snippet = c.get('text', '')[:200].replace('\n', ' ')
        candidates_text += (
            f"[{i}] Artist: {c.get('artist', 'N/A')}, "
            f"Album: {c.get('album', 'N/A')}, "
            f"Score: {c.get('score', 'N/A')}, "
            f"Genre: {c.get('genre', 'N/A')}\n"
            f"    Excerpt: {snippet}\n\n"
        )

    rerank_prompt = (
        f'You are ranking album review candidates by how well they answer the user query.\n'
        f'Return ONLY a JSON array of candidate indices ordered from most to least relevant.\n'
        f'Example: [2, 0, 4, 1, 3]\n\n'
        f'Query: {query}\n\n'
        f'Candidates:\n{candidates_text}'
    )

    response = client.responses.create(
        model=MODEL,
        instructions='Return only a valid JSON array of integers. No explanation.',
        input=[{'role': 'user', 'content': rerank_prompt}],
        max_output_tokens=100,
        temperature=0.0
    )

    try:
        raw = response.output_text.strip()
        # Extract the JSON array even if the model wraps it in markdown
        start = raw.find('[')
        end = raw.rfind(']') + 1
        ranked_indices = json.loads(raw[start:end])
        # Filter valid indices and deduplicate
        valid = [i for i in ranked_indices if isinstance(i, int) and 0 <= i < len(context_data)]
        seen = set()
        deduped = [i for i in valid if not (i in seen or seen.add(i))]
        reranked = [context_data[i] for i in deduped]
        # Append any missing candidates at the end
        missing = [c for i, c in enumerate(context_data) if i not in seen]
        return (reranked + missing)[:top_k]
    except Exception:
        # Fallback: return original order
        return context_data[:top_k]

In [ ]:
# Retrieve top-10 candidates, then re-rank to top-3
candidates = get_context_data(SAMPLE_QUERY, collection, top_n=10)

print('--- Before re-ranking (top 3 of 10 by vector similarity) ---')
for i, r in enumerate(candidates[:3]):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

reranked = llm_rerank(candidates, SAMPLE_QUERY, top_k=3)

print('\n--- After re-ranking ---')
for i, r in enumerate(reranked):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

In [ ]:
reranked_response = generate_prompt(SAMPLE_QUERY, reranked)
final_reranked = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': reranked_response}],
    max_output_tokens=500,
    temperature=0.7
).output_text
display(Markdown(final_reranked))

**Cost trade-off:**

Re-ranking adds one extra LLM call — but it is a *short* call: the input is a compact candidate list and the output is just a list of indices. At course scale (10 candidates, ~200 chars each), this is roughly 300–500 tokens, which is negligible.

At production scale with thousands of candidates, you would use a dedicated re-ranking model (e.g., Cohere Rerank, cross-encoders) rather than a generative LLM — but the principle is the same.

---

## Section 5: Combined Pipeline (`hybrid_rag`)

Now we compose all three techniques into a single function:

1. **Keyword filter** (`where_document`) — restrict to documents mentioning the keyword
2. **Metadata filter** (`where`) — enforce quality or recency constraints
3. **LLM re-ranking** — promote the most relevant from the narrowed candidate set

Each layer trades **recall** for **precision**: the keyword and metadata filters reduce the candidate pool, then re-ranking refines the order. The net effect is higher-quality results for queries where the constraints are meaningful.

In [ ]:
def hybrid_rag(
    query: str,
    collection: chromadb.api.models.Collection,
    keyword: str = None,
    where_filter: dict = None,
    top_n_candidates: int = 10,
    top_k_final: int = 3
) -> list:
    """
    Hybrid retrieval pipeline:
      1. ChromaDB vector search with optional keyword + metadata filters
      2. LLM re-ranking to top_k_final
    Returns a list of context_data dicts.
    """
    where_document = {'$contains': keyword} if keyword else None
    candidates = get_context_data(
        query, collection, top_n_candidates,
        where_document=where_document,
        where=where_filter
    )
    if not candidates:
        print('Warning: no candidates returned for the given filters.')
        return []
    return llm_rerank(candidates, query, top_k=top_k_final)

**Query 1:** Highly rated indie albums — keyword + score filter + re-ranking

In [ ]:
results_1 = hybrid_rag(
    query=SAMPLE_QUERY,
    collection=collection,
    keyword='indie',
    where_filter={'score': {'$gt': 8.0}},
    top_n_candidates=10,
    top_k_final=3
)

response_1 = generate_prompt(SAMPLE_QUERY, results_1)
answer_1 = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': response_1}],
    max_output_tokens=500,
    temperature=0.7
).output_text
display(Markdown(answer_1))

**Query 2:** Classic albums from the 90s — year filter + re-ranking

In [ ]:
query_2 = 'What are the most influential rock albums of the 1990s?'

results_2 = hybrid_rag(
    query=query_2,
    collection=collection,
    keyword='rock',
    where_filter={'$and': [{'score': {'$gt': 7.5}}, {'year': {'$gte': 1990}}, {'year': {'$lt': 2000}}]},
    top_n_candidates=10,
    top_k_final=3
)

response_2 = generate_prompt(query_2, results_2)
answer_2 = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': response_2}],
    max_output_tokens=500,
    temperature=0.7
).output_text
display(Markdown(answer_2))

**Query 3:** Electronic music — keyword only + re-ranking (no metadata filter)

In [ ]:
query_3 = 'Recommend some groundbreaking electronic albums with experimental production.'

results_3 = hybrid_rag(
    query=query_3,
    collection=collection,
    keyword='electronic',
    where_filter=None,
    top_n_candidates=10,
    top_k_final=3
)

response_3 = generate_prompt(query_3, results_3)
answer_3 = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': response_3}],
    max_output_tokens=500,
    temperature=0.7
).output_text
display(Markdown(answer_3))

---

## Summary

| Technique | What it does | Best for | Risk |
|-----------|-------------|----------|------|
| **Pure vector** | Semantic similarity across full corpus | Open-ended queries | May retrieve semantically close but unhelpful chunks |
| **Keyword filter** | Restricts to documents containing an exact term | Artist names, genre labels, specific albums | Misses synonyms; empty result if keyword is rare |
| **Metadata filter** | Restricts to chunks meeting numeric/string criteria | Quality gates (score), time ranges (year) | Tight filters reduce recall |
| **LLM re-ranking** | Model re-orders a candidate pool by relevance | Refining any of the above | Extra API call; negligible cost at small scale |
| **Hybrid (all three)** | Combines all layers | High-precision queries with clear constraints | Very tight keyword + metadata may return no candidates |

**Next steps:**
- Notebook `02_9` (coming soon): quantitative evaluation — measure how much each technique improves hit rate
- Try query intent extraction: use the LLM to parse `genre`, `score`, and `year` constraints from a natural-language query, then pass them as `where_filter` automatically